# Milestone 1 — 把 IESO 小時需求資料落地這個 notebook 不放邏輯,只負責**驅動** repo 裡的腳本。真正的程式碼在 `src/ingest/`,這樣 Colab 跟本機跑出來的東西才會是同一個。**每次連上新的 Colab session,從第 1 格依序跑到第 4 格。**資料存在 Google Drive,不會因為 session 斷線而消失。| 東西 | 放哪 | 會不會消失 ||---|---|---|| 程式碼 | GitHub,每次 clone 到 `/content/` | 會,但重 clone 就有 || 原始 CSV / Parquet | Google Drive | 不會 || `reports/` 輸出 | repo 裡,第 8 格會複製一份到 Drive | 複製過就不會 |

## 1. 環境

In [ ]:
!pip install -q polars duckdbfrom google.colab import drivedrive.mount('/content/drive')

## 2. 取得程式碼第一次跑會 clone,之後每次跑會 pull 最新版。> 在 GitHub 網頁上改完 `src/` 裡的檔案之後,回來重跑這一格就會拿到新版。

In [ ]:
import os, pathlib, subprocessREPO_URL = "https://github.com/ethantosc/ieso-demand-forecast.git"REPO_DIR = pathlib.Path("/content/ieso-demand-forecast")if (REPO_DIR / ".git").exists():    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)else:    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)os.chdir(REPO_DIR)print("cwd:", pathlib.Path.cwd())print(subprocess.run(["git", "log", "-1", "--oneline"],                     capture_output=True, text=True).stdout.strip())

## 3. 把 `data/` 指到 Driverepo 裡的 `data/` 換成一個指向 Drive 的捷徑。這樣 `src/` 裡的腳本完全不用知道自己跑在 Colab 上——它們看到的還是 `data/raw/demand/`,跟本機一模一樣。安全閥:如果 repo 的 `data/` 裡已經有真的資料檔,這一格會停下來,不會覆蓋。

In [ ]:
import pathlib, shutilDRIVE_DATA = pathlib.Path("/content/drive/MyDrive/ieso-demand-forecast-data")for sub in ("raw/demand", "raw/forecasts", "staging", "curated"):    (DRIVE_DATA / sub).mkdir(parents=True, exist_ok=True)local = pathlib.Path("data")if local.is_symlink():    local.unlink()elif local.exists():    payload = [p for p in local.rglob("*") if p.suffix.lower() in {".csv", ".parquet", ".xml"}]    if payload:        raise SystemExit(            f"repo 的 data/ 裡有 {len(payload)} 個資料檔,先確認要不要留,再手動處理。"        )    shutil.rmtree(local)local.symlink_to(DRIVE_DATA)print("data ->", local.resolve())print("已有的原始檔:", len(list((DRIVE_DATA / "raw/demand").glob("*.csv"))), "個")

## 4. 下載原始 CSV2002 到今年的年度檔。已經抓過的年度會跳過,只有當年度那支每次都重抓(因為它還在長)。`manifest.json` 會記下每個檔的下載時間、大小、SHA-256。如果某個「已定版」的年度檔哪天內容變了,下次 `--force` 重抓時會印出 `[CONTENT CHANGED]`。

In [ ]:
!python -m src.ingest.fetch_demand

## 5. 落成 Parquet讀年度 CSV → 處理 hour-ending 1–24 → 寫 `data/staging/demand_hourly.parquet`。原始檔不會被修改,所以這一格永遠可以重跑。

In [ ]:
!python -m src.ingest.demand

## 6. 驗證這一格**會失敗,而且失敗是有意義的**。它在測試 ingest 所依賴的假設:時間軸是不是固定 EST、每天是不是完整、序列有沒有斷。- exit 0 → 必須為空的檢查都過了- exit 1 → Parquet 不可信,不要在上面做特徵工程輸出寫到 `reports/data_quality_auto.md`。

In [ ]:
!python -m src.ingest.check_demand

## 7. 看結果下面整份貼出來。人要看的是這幾節:- **§2 / §7** 固定 EST 有沒有成立(§7 應該剛好每年 2 個日期:3 月 23 小時、11 月 25 小時)- **§6** 年度檔接縫有沒有跳階,特別是 2002→2003- **§12 / §13** Market 減 Ontario 的差額

In [ ]:
print(pathlib.Path("reports/data_quality_auto.md").read_text())

## 8. 存一份到 Driverepo 的 clone 會隨 session 消失,所以把報告複製出去。之後要進 git 的話,從 Drive 下載那個 `.md`,用 GitHub 網頁的 **Add file → Upload files**傳到 `reports/`。

In [ ]:
import shutil, pathlibdest = pathlib.Path("/content/drive/MyDrive/ieso-demand-forecast-data/reports")dest.mkdir(parents=True, exist_ok=True)shutil.copy("reports/data_quality_auto.md", dest / "data_quality_auto.md")print("saved ->", dest / "data_quality_auto.md")